

<div style="max-width: 850px;">

# **LLM - The Transformer lifecycle**

---

This project is used to understand the lifecycle of transformers 
From data cleaning to speed up inference
We will keep it small such that we can train several transformers
All will be implemented from scratch
Later we can upgrade to use everything using hugging face interfaces


**Requirements**

These are requirements and how to install them in your environment

* `pip install datasets`

</div>



<div style="max-width: 850px;">

# The dataset

---

## Data curation and preparation

---

We will start small, using `tiny stories` dataset.
This dataset lives in huggingface. 

We will observe it's distribution and compute useful statistics from the dataset.

To process text we should also use a tokenizer. At the moment we are not interested in training one ourselves and we will use the `GPT2TokenizerFast` available in Huggingface. `GPT2TokenizeFast` implements a BPE (byte-pair encoding) tokenization.

</div>


In [ ]:
%load_ext autoreload
%autoreload 2

import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_from_disk
from IPython.display import display
from transformers import GPT2TokenizerFast

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

import data_cycle
import analysis_results as analysis
import diagnostics as diagnostic

logging.basicConfig(level=logging.INFO)
logging.getLogger('httpx').setLevel(logging.ERROR)
logging.getLogger('httpcore').setLevel(logging.ERROR)
_LOGGER = logging.getLogger(__name__)

# Select the artifact folder and configuration for this analysis.
artifacts_path = Path('/home/angel/src/transformers_lifecycle/artifacts/dataset_cycle')
data_path = project_root / 'datasets'
config_path = project_root / 'configs' / 'dataset_cycle.json'
pipeline_config = json.loads(config_path.read_text())
num_proc = pipeline_config['num_proc']
tokenization_batch_size = 512
train_source = data_cycle.TINYSTORIES_TRAIN_SOURCE
valid_source = data_cycle.TINYSTORIES_VALIDATION_SOURCE


**Generate normalized datasets (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_dataset_preparation.py --output-dir artifacts/dataset_cycle
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


In [5]:
# Load saved normalized, nonempty stories; no raw-dataset download is needed.
stories_train_dataset = load_from_disk(str(artifacts_path / 'normalized_train'))
stories_valid_dataset = load_from_disk(str(artifacts_path / 'normalized_validation'))


In [4]:
# Preview normalized training documents (these are not the raw source texts).
_LOGGER.info('Previewing normalized training documents after empty removal')
data_cycle.display_dataset_information(stories_train_dataset)
display(stories_train_dataset.select(range(min(10, len(stories_train_dataset)))).to_pandas())


INFO:__main__:Previewing normalized training documents after empty removal
INFO:diagnostics:Columns (1): ['text']
INFO:diagnostics:Rows: 2119489
INFO:diagnostics:Features: {'text': Value('string')}
INFO:diagnostics:Split: None
INFO:diagnostics:Cache files: [{'filename': '/Users/angelmartinez/src/transformers_lifecycle/artifacts/dataset_cycle/normalized_train/data-00000-of-00004.arrow'}, {'filename': '/Users/angelmartinez/src/transformers_lifecycle/artifacts/dataset_cycle/normalized_train/data-00001-of-00004.arrow'}, {'filename': '/Users/angelmartinez/src/transformers_lifecycle/artifacts/dataset_cycle/normalized_train/data-00002-of-00004.arrow'}, {'filename': '/Users/angelmartinez/src/transformers_lifecycle/artifacts/dataset_cycle/normalized_train/data-00003-of-00004.arrow'}]
INFO:diagnostics:Sample 1: {'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so sh

,text
0,"One day, a little girl named Lily found a need..."
1,"Once upon a time, there was a little car named..."
2,"One day, a little fish named Fin was swimming ..."
3,"Once upon a time, in a land full of trees, the..."
4,"Once upon a time, there was a little girl name..."
5,"Once upon a time, in a big lake, there was a b..."
6,"Once upon a time, in a small town, there was a..."
7,"Once upon a time, in a peaceful town, there li..."
8,"Once upon a time, there was a clever little do..."
9,"One day, a fast driver named Tim went for a ri..."



<div style="max-width: 850px;">

## Dataset diagnosis
---

We run data diagnosis by computing and observing dataset statistics. These will help us to decide what pre-processing steps we need to apply to clean, standrize and filter documents. 

Usually this step is performed on a tokenized dataset. We will keep it simple and use `GPT2TokenizerFast`, which is a tokenizer trained using the BytePairing algorithm, and compute the following:

   - **Number of documents**: a document can be simply put as the basic unit in the dataset. This is a story within `tiny stories` dataset.
   - **Document length distribution**: the distribution help us identify documents that are too short or too long. 
   - **Vocabulary/Token distribution**: we will plot Zipf plots. These help us to identify if there are tokens that are too often repeating - such as html tags - and can indicate garbage or spurious dataset. For more context on these plots look [at this article.](https://www.nngroup.com/articles/zipf-curves-and-website-popularity/#:~:text=According%20to%20much%20available%20data%2C%20web%20use,number%20of%20elements%20that%20score%20very%20low)

**Diagnosis runs separately:** run `python scripts/run_dataset_diagnosis.py` from the project root, then load its artifacts below. No tokenized dataset is saved. Diagnosis counts full token sequences without truncation; GPT-2 training sequences are limited separately.

</div>



**Generate diagnosis artifacts (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_dataset_diagnosis.py --output-dir artifacts/dataset_cycle/diagnosis
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


In [ ]:
document_lengths, token_frequencies, diagnosis_summary = analysis.load_diagnosis(
    artifacts_path / 'diagnosis',
)
sequence_lengths = document_lengths['token_count'].to_numpy()
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2', local_files_only=True)


In [6]:
vocabulary = tokenizer.get_vocab()
_LOGGER.info('Len of vocabulary: %d', len(vocabulary))
_LOGGER.info('Loaded diagnosis for %d documents', diagnosis_summary['document_count'])


INFO:__main__:Len of vocabulary: 50257
INFO:__main__:Loaded diagnosis for 2119719 documents


In [7]:
# Reconstruct the inputs used by the existing plots from saved counts.
id_dist = dict(zip(token_frequencies['token_id'], token_frequencies['count']))
id_dist_list = sorted(id_dist.items())


In [8]:
# Looking to some parts of the dataset

# Let's look at the largest ocurrence. We will find that there is only a few
# tokens that are taking a lot of the counts -> over 10M occurences!
topk = 10
id_dist__ = sorted([(k, v) for (k, v) in id_dist_list], key=lambda x: x[1])

_LOGGER.info(f' The following are the {topk} tokens with largest ocurrence')
vocab_reversed = {v:k for k, v in vocabulary.items()}

for k, v in [[vocab_reversed[x[0]], x[1]] for x in id_dist__[-topk:]]:
    _LOGGER.info(f' {k} -> {v}')


INFO:__main__: The following are the 10 tokens with largest ocurrence
INFO:__main__: Ġher -> 4523829
INFO:__main__: Ġit -> 5185957
INFO:__main__: Ġwas -> 9437796
INFO:__main__: Ġa -> 11368431
INFO:__main__: Ġto -> 12634330
INFO:__main__: Ġthe -> 16600383
INFO:__main__: , -> 16631454
INFO:__main__: Ġand -> 17810413
INFO:__main__: Ċ -> 18424730
INFO:__main__: . -> 34651025


In [ ]:
display(pd.Series(diagnosis_summary, name='Raw training diagnosis'))


In [ ]:
fig, axes = diagnostic.plot_dataset_diagnosis(
    sequence_lengths, id_dist, diagnosis_summary, tokenizer,
)
plt.show()


In [11]:
# Observing the most and least ranked tokens
N = 10  # configurable

# Sort (token_id, count) by frequency
ranked_tokens = sorted(
    id_dist.items(),
    key=lambda x: x[1],
    reverse=True
)

def show_token(tokenizer, token_id):
    # Raw GPT-2 token representation + decoded human-readable text
    raw = tokenizer.convert_ids_to_tokens(token_id)
    decoded = tokenizer.decode([token_id])
    return raw, decoded


print(f"Top {N} highest-ranked tokens:")
for rank, (token_id, count) in enumerate(ranked_tokens[:N], start=1):
    raw, decoded = show_token(tokenizer, token_id)
    print(
        f"Rank {rank:>6} | "
        f"ID {token_id:>6} | "
        f"token={repr(raw):<18} | "
        f"text={repr(decoded):<18} | "
        f"count={count}"
    )

print(f"\nBottom {N} lowest-ranked tokens:")
total = len(ranked_tokens)

for rank, (token_id, count) in enumerate(
    ranked_tokens[-N:],
    start=total - N + 1
):
    raw, decoded = show_token(tokenizer, token_id)
    print(
        f"Rank {rank:>6} | "
        f"ID {token_id:>6} | "
        f"token={repr(raw):<18} | "
        f"text={repr(decoded):<18} | "
        f"count={count}"
    )

Top 10 highest-ranked tokens:
Rank      1 | ID     13 | token='.'                | text='.'                | count=34651025
Rank      2 | ID    198 | token='Ċ'                | text='\n'               | count=18424730
Rank      3 | ID    290 | token='Ġand'             | text=' and'             | count=17810413
Rank      4 | ID     11 | token=','                | text=','                | count=16631454
Rank      5 | ID    262 | token='Ġthe'             | text=' the'             | count=16600383
Rank      6 | ID    284 | token='Ġto'              | text=' to'              | count=12634330
Rank      7 | ID    257 | token='Ġa'               | text=' a'               | count=11368431
Rank      8 | ID    373 | token='Ġwas'             | text=' was'             | count=9437796
Rank      9 | ID    340 | token='Ġit'              | text=' it'              | count=5185957
Rank     10 | ID    607 | token='Ġher'             | text=' her'             | count=4523829

Bottom 10 lowest-ranked tokens:



<div style="max-width: 850px;">

**Diagnosis**


* **Most frequent tokens**. These are punctuation tokens and lots of short length words. For training, keep punctuation and frequent tokens - **removing them breaks language modeling because the model must learn *syntax, spacing, sentence boundaries, formatting, etc.***. **It only becomes a problem** if the top tokens are things like `<, >, +, http, www, etc` or repeated boilerplate tokens from HTML, templates, logs, menus, tables, or corrupted text.

    * **Conclusion**. There are no unwanted tokens in this dataset. When removed punctuation and `articles` such as "the", "that" etc, most frequent tokens become pronouns like "he", "she" - which is normal in stories.


* **Length of documents have long tails**. We don't want documents on both tails - very short documents and very long documents. The length of the documents do not make them "be of bad quality", but it's more of a question of efficient training. For example, very short documents make training inefficient because **the model sees little context**. Short examples often include: titles, menu labels, broken web fragments, captions, etc. Very long documents will simply be cut by a `MAX_SEQUENCE_LENGTH` - loading them into memory will be inefficient. In practice document length will be kept in boundaries `MIN_SEQUENCE_LENGTH` and `MAX_SEQUENCE_LENGTH` with boundaries depending on the dataset.

    * **Conclusion**. There is a long tail for long documents, but overall the distribution looks good. During our exercise we will keep the length of ducuent short with `MIN_SEQUENCE_LENGTH=128` and `MAX_SEQUENCE_LENGTH=512`.


* **Zipf curves**. Zipf curves help us visualizaing the distributions of the tokens in the dataset by ranking: most frequent token has rank 1. This plot should be looking smooth and decreasing and spikes and clifts on the tail. We can diagnose the curve by visualizaing the ideal Zipf distribution `1/rank`.

    * **Conclusion**. The token-frequency distribution approximately follows a Zipf-like distribution, with token frequency decreasing roughly as a power law with rank. The highest-frequency tokens include punctuation, whitespace-related tokens, and common lexical/subword tokens, which explains the relatively flat head of the distribution. The middle portion of the distribution follows the 1/rank reference reasonably well. At high ranks, the empirical distribution falls below the ideal Zipf curve, forming a long tail of rare tokens. Inspection of these tokens shows meaningful low-frequency terms such as "Bigfoot" and "Iodine", suggesting that the tail contains legitimate story-specific vocabulary rather than simply noise. Therefore, rare tokens should not be removed solely based on their frequency.


</div>



<div style="max-width: 850px;">

## Pre-processing and data curation
---

**Data curation is the first and upmost component on quality control on LLMs**. Curation is performed as an iterative process where apply filters, models and heuristics to improve the overall quality. Common pre-processing steps are removing exact duplicates, removing semantic similar elements, and filter elements based on "explainability". Each step in the data curation cycle contributes in different ways to efficient learning of LLMs. 

Data curation follows a simple recipe:

1. **Normalise text**: unicode conversion, boilerplate, stripping, etc,
2. **Remove empty documents**: filter empty or whitespace-only text after normalization,
3. **Exact deduplication**: using hasing,
4. **Near deduplication**: using minhash and local sensitive hashing (LSH),
5. **Apply tokenization**, we can use the same BPE tokenizer,
6. **Filter by token length**: after tokenization we remove elements outside of `MIN_SEQUENCE_LENGTH` and `MAX_SEQUENCE_LENGTH`,
7. **Perplexity filtering**: using document "explainability" from a LLM as a meassure of the quality (can be a strong or a small LLM),
8. **[optional] Semantic deduplication**: converting documents to embeddings and using nearest neigbours,
9. **Iteratively curate** dataset by training and analysing outputs.


In the following we will implement each of these steps in our dataset.

</div>




<div style="max-width: 850px;">

### Step 1 - Normalizing text

---

In the normalization step we simply convert all the text to unicode, strip and remove more than 1 spaces. Specifically, we will focus in the following normalization

1. **Unicode normalization**. Normalize different unicode representations into a canonical form: curly quotes → standard quotes, accented characters → consistent encoding,
2. **Whitespace normalization**. Collapse multiple spaces, normalize tabs/newlines, strip leading/trailing spaces,
3. **Line break normalization**. Convert `\r\n`, `\r → \n` and collapse excessive newlines (e.g., >2 → 2),
4. **Control character removal**. Remove: non-printable characters, zero-width spaces and Unicode artifacts,
5. **Standardize quotes and punctuation**. “ ” → ", ‘ ’ → ', normalize dashes (— → -),
6. **Lowercasing reduces vocabulary entropy, but removes signal (e.g., named entities)** so it's better to keep casing.

Run `python scripts/run_dataset_preparation.py` from the project root to normalize training and validation data and remove empty documents. The following cells load the saved datasets and cleaning metrics; no normalization is repeated in the notebook.


</div>


In [ ]:
# Normalized datasets were loaded above for previews and remain unchanged.
# Load the saved cleaning metrics; do not rerun normalization or filtering.
cleaning_metrics = pd.read_csv(artifacts_path / 'cleaning_metrics.csv')


In [13]:
display(cleaning_metrics)
for row in cleaning_metrics.itertuples(index=False):
    _LOGGER.info(
        '%s: %d documents after normalization; %d after empty removal (%.2f%% removed)',
        row.split, row.after_normalization, row.after_empty_removal, row.removed_percentage,
    )


,split,input_count,after_normalization,after_empty_removal,removed_count,removed_percentage
0,Training,2119719,2119719,2119489,230,0.01085
1,Validation,21990,21990,21990,0,0.00000


INFO:__main__:Training: 2119719 documents after normalization; 2119489 after empty removal (0.01% removed)
INFO:__main__:Validation: 21990 documents after normalization; 21990 after empty removal (0.00% removed)



<div style="max-width: 850px;">


### Step 2 -Exact deduplication with hashing

---

We will use a hash to remove exact duplicates. First, we will convert all the text to a hash - after normalization - using `hashlib.sha256`. This will create a set of hashes whifh to perform deduplication and make fast lookups.

</div>

Run `python scripts/run_exact_deduplication.py` to hash and deduplicate the saved normalized datasets. Load the results below to plot removal percentages and inspect duplicate pairs.


**Generate exact-deduplication artifacts (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_exact_deduplication.py --artifacts-dir artifacts/dataset_cycle
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


In [ ]:
stories_train_dataset_cpy = load_from_disk(str(artifacts_path / 'normalized_train'))
stories_train_dataset = load_from_disk(str(artifacts_path / 'exact_dedup_train'))
stories_valid_dataset = load_from_disk(str(artifacts_path / 'exact_dedup_validation'))


In [ ]:
exact_dedup_metrics = pd.read_csv(artifacts_path / 'exact_dedup_metrics.csv')
display(exact_dedup_metrics)


In [ ]:
X = 10
exact_duplicate_examples = analysis.load_exact_duplicate_pairs(
    artifacts_path / 'exact_duplicate_pairs.csv', count=X,
)


In [ ]:
diagnostic.display_duplicate_pairs(
    stories_train_dataset_cpy,
    zip(exact_duplicate_examples['original_index'], exact_duplicate_examples['duplicate_index']),
    text_column=train_source.text_column, max_pairs=X,
)


In [ ]:
fig, ax = diagnostic.plot_exact_deduplication(exact_dedup_metrics)
plt.show()



<div style="max-width: 850px;">


### Step 3 - Near deduplication

---

Finding near duplicates help us to remove documents in the dataset that are not exact duplicates based on text, but instead have similar structure and content. We can define that two documents are near duplicates if they contain similar local patterns in their `n-grams` that are usually constructed in word level. 

N-grams have the following properties:

1. Preserve local structure,
2. Are robust to small details,
3. Ignore global ordering,
4. Are sensitive to similar/copied structure.

Using the N-grams of two documents, the similarity becomes comparing how much overlap have the two sets:

$$
J(A,B) = \frac{|A\cap B|}{|A\cup B|}.
$$

The above is the *Jaccard similarity*. However, at scale this becomes infeasible given that the complexity is $O(N^2)$, $N$ equal to the length of both sets. The solution is to use **MinHashing**. MinHashing converts the documents to a signature and allows us to compute the above expression with the property:

$$
P(\text{MinHash}(A)=\text{MinHash}(B)) = J(A,B).
$$

**The intuition** behind is that, for a permutation of an n-gram that is in $A\cup B$ it must be in: 1) in $A\cap B$, or 2) must be in A or B. The probability that lies in the intersection is $\frac{|A\cap B|}{|A\cup B|}$.

To implement **MinHashing** we simulate permutations with hash functions $h_i$ to create a document signature:

$$
m_i = h_i(g), g\in A.
$$

Then

$$
\text{signature}(A) = [m_1, m_2, m_3, ..., m_K],
$$

for $k=K$ hash functions. Then we compare two documents by computing:

$$
\hat{J}(A, B) = \frac{1}{K} \sum_{i=1}^K I[m_i(A)=m_i(B)].
$$

However, this computation is still $O(N^2)$. To speed up computation we turn our eyes to LSH (local sensitive hashing) to **help us to only compute the expensive comparison metric on documents that are similar in some bands**. Specifically, LSH performs the following steps:

1. Split each MinHash signature into $b$ bands, each containing $r$ rows:

$$
n = b\cdot r,
$$

where \(n\) is the signature length.

2. Hash each band independently into a hash table/bucket.

3. Define candidate pairs: two documents $A,B$ become a candidate pair if their signatures are identical in all $r$ rows of at least one band.

4 Only for these candidate pairs, compute a more precise similarity measure, such as $\hat{J}(A, B)$.


For example, for a two component band, a signature can become:
   
$$
\text{signature}(A) = [[m_1, m_2], [m_3, m_4], ..., [m_{K-1}, m_K]].
$$

Luckily, there are packages in Python that can help us computing these permutations - MinHashes and similarities with LSH. In this exercise we will use `datasketch`. In practice, we need to tune the following parameters:

- **`NGRAM size` — controls the representation.** We split each document into overlapping $n$-grams before computing MinHash. Smaller $n$-grams create more overlap between documents and are more tolerant of small textual changes, but can make unrelated documents appear similar. Larger $n$-grams capture more specific context and reduce accidental matches, but become more sensitive to small edits. **Trade-off:** smaller $n$ → higher recall / lower precision; larger $n$ → higher precision / lower recall.

- **`Number of permutations` — controls the accuracy of MinHash.** This determines the number of hash functions used to construct each MinHash signature. More permutations give a more accurate estimate of the true Jaccard similarity and reduce the variance of $\hat J(A,B)$, but increase hashing cost, memory usage, and LSH index size. **Trade-off:** more permutations → better similarity estimates but more compute and memory; fewer → faster and cheaper but noisier estimates.

- **`Similitud threshold` — controls which similarities LSH targets.** This tells LSH approximately where we want the transition between unlikely and likely candidate pairs to occur. A higher threshold retrieves fewer, more strongly similar candidates but risks missing useful moderately similar pairs. A lower threshold generates more candidates, improving recall but increasing false positives and downstream comparisons. **Trade-off:** higher threshold → higher precision / lower recall; lower threshold → higher recall / lower precision.

- **\(b\) and \(r\) — control the LSH banding behavior.** The MinHash signature is divided into \(b\) bands containing $r$ rows each, roughly satisfying $br=\text{NUM\_PERM}$. Two documents become candidates when they match on all \(r\) rows of at least one band. More bands with fewer rows make LSH more permissive; fewer bands with more rows make it stricter. **Trade-off:** more bands → higher recall / more false positives; more rows per band → higher precision / more false negatives.


</div>


### Compare saved n-gram experiments

Load both experiments below without rerunning deduplication. Compare retrieved LSH candidates and removed documents against the training exact-deduplication baseline, then preview removed documents using saved retained indices.


**Generate signatures before exact deduplication (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_signatures.py --input-dataset artifacts/dataset_cycle/normalized_train --output-dir artifacts/dataset_cycle/normalized_train_signatures --ngram-sizes 2 3 4 5
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


**Run n-gram sweep before exact deduplication (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_dedup_ngram_experiments.py --config configs/dataset_cycle.json --input-dataset artifacts/dataset_cycle/normalized_train --signatures-dir artifacts/dataset_cycle/normalized_train_signatures --output-dir artifacts/dataset_cycle/normalized_train_ngram_experiment
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


**Generate signatures after exact deduplication (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_signatures.py --input-dataset artifacts/dataset_cycle/exact_dedup_train --output-dir artifacts/dataset_cycle/experiment_exact_and_minhash/signatures --ngram-sizes 2 3 4 5
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


**Run n-gram sweep after exact deduplication (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_dedup_ngram_experiments.py --config configs/dataset_cycle.json --input-dataset artifacts/dataset_cycle/exact_dedup_train --signatures-dir artifacts/dataset_cycle/experiment_exact_and_minhash/signatures --output-dir artifacts/dataset_cycle/experiment_exact_and_minhash/results
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


Candidate totals are loaded directly from result CSVs. If `minhash_candidate_counts.json` exists, its historical corrected values are used for the comparison. No separate generation step is required.


In [ ]:
ngram_result_dirs = {
    'With exact duplicates': artifacts_path / 'normalized_train_ngram_experiment',
    'Without exact duplicates': artifacts_path / 'experiment_exact_and_minhash' / 'results',
}
ngram_results_table = analysis.load_ngram_comparison(
    ngram_result_dirs, candidate_counts_path=artifacts_path / 'minhash_candidate_counts.json',
)
if ngram_results_table is not None:
    display(ngram_results_table[[
        'dataset', 'ngram_size', 'total_documents', 'candidate_total',
        'removed_count', 'removed_percentage',
    ]])


In [ ]:
if ngram_results_table is not None and not ngram_results_table.empty:
    exact_training_removed = int(exact_dedup_metrics.set_index('split').loc['Training', 'removed_count'])
    fig, axes = diagnostic.plot_ngram_comparison(
        ngram_results_table, labels=ngram_result_dirs, exact_training_removed=exact_training_removed,
    )
    plt.show()


In [ ]:
preview_ngram = 2
preview_limit = 5
ngram_input_datasets = {
    'With exact duplicates': artifacts_path / 'normalized_train',
    'Without exact duplicates': artifacts_path / 'exact_dedup_train',
}
if ngram_results_table is not None:
    for label, directory in ngram_result_dirs.items():
        print(f'{label}, {preview_ngram}-gram: removed document preview')
        diagnostic.display_document_table(analysis.load_removed_documents(
            directory, preview_ngram, ngram_input_datasets[label], count=preview_limit,
        ))


#### Experiment 2: Vary similarity threshold

Run independently from the production pipeline:

```bash
python scripts/run_minhash_dedup_threshold_experiments.py --config configs/dataset_cycle.json
```

The script reuses saved signatures for the single n-gram size in `threshold_experiment.ngram_size`, varies the configured similarity thresholds, and keeps the LSH threshold fixed. No signatures are recomputed. The following cells load `threshold_results.csv` and plot the saved results. Missing optional results are skipped. Add `--overwrite` to rerun completed experiments.


**Run similarity-threshold sweep (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_dedup_threshold_experiments.py --config configs/dataset_cycle.json --input-dataset artifacts/dataset_cycle/exact_dedup_train --signatures-dir artifacts/dataset_cycle/experiment_exact_and_minhash/signatures --output-dir artifacts/dataset_cycle/experiment_exact_and_minhash/minhash_similarity_threshold_results
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


In [ ]:
threshold_results_dir = artifacts_path / 'experiment_exact_and_minhash' / 'minhash_similarity_threshold_results'
threshold_ngram = 2
threshold_results_table = analysis.load_experiment_results(
    threshold_results_dir, kind='threshold', ngram_size=threshold_ngram,
)
if threshold_results_table is not None:
    display(threshold_results_table)


In [ ]:
if threshold_results_table is not None and not threshold_results_table.empty:
    fig, ax = diagnostic.plot_threshold_results(threshold_results_table)
    plt.show()


**Regenerate matched pairs for one threshold (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_dedup_threshold_experiments.py --config configs/dataset_cycle.json --input-dataset artifacts/dataset_cycle/exact_dedup_train --signatures-dir artifacts/dataset_cycle/experiment_exact_and_minhash/signatures --output-dir artifacts/dataset_cycle/experiment_exact_and_minhash/minhash_similarity_threshold_results --thresholds 0.1 --overwrite
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


In [ ]:
inspect_threshold = 0.1
X = 10
matched_pairs = None
if threshold_results_table is not None and not threshold_results_table.empty:
    pair_result = analysis.load_duplicate_pairs(threshold_results_dir, f'threshold_{inspect_threshold:g}')
    if pair_result is not None:
        matched_pairs, pair_metadata = pair_result
        pair_source_dataset = analysis.load_source_dataset(pair_metadata)
        diagnostic.display_similarity_extremes(
            pair_source_dataset, matched_pairs, count=X, text_column=pair_metadata['text_column'],
        )


In [ ]:
if matched_pairs is not None:
    fig, ax = diagnostic.plot_matched_similarity_distribution(matched_pairs, inspect_threshold)
    plt.show()


**Generate production MinHash artifacts (only if needed).** Run from the project root; replace `artifacts/dataset_cycle` if using another artifact folder.

```bash
python scripts/run_minhash_deduplication.py --config configs/dataset_cycle.json
```

Completed experiment/signature results are reused. Add `--overwrite` when deliberately regenerating them.


In [ ]:
stories_train_dataset = load_from_disk(str(artifacts_path / 'minhash_dedup_train'))
stories_valid_dataset = load_from_disk(str(artifacts_path / 'minhash_dedup_validation'))


In [ ]:
minhash_dedup_metrics = pd.read_csv(artifacts_path / 'minhash_dedup_metrics.csv')
display(minhash_dedup_metrics)
for row in minhash_dedup_metrics.itertuples(index=False):
    _LOGGER.info(
        '%s: %d -> %d documents (%.2f%% removed by MinHash)',
        row.split, row.before, row.after, row.removed_percentage,
    )




<div style="max-width: 850px;">

### Step 4 - Tokenization

---

A dataset consist of documents. However, dataset is usually regarded as amount of tokens it cointains. Even more, when training a model we usually refer to the total count of tokens the model saw during training considering epochs too - e.g. GPT4 was trained on XX tokens. GPT models use a **Byte-Pair Encoding (BPE)** tokenizers, which decompose words into sub-words based on the most frequent pair of characters in subwords.

As an example, consider the input dataset `'low', 'lower', 'lowest'`. BPE works as follows

1. **Initialize vocabulary** - start with all unique characters as tokens including a token for unknown:

Vocabulary: `'l', 'o', 'w', 'e', 'r', 's', 't', '<unk>'`

2. **Count token pairs in the dataset**

```
(l,o) = 3    (w,e) = 2   (e,s) = 1
(o,w) = 3   (e,r) = 2   (s,t) = 1
```

3. **Merge most frequent pair into new subword token**:

Vocabulary after merging `(l,o)`: `'lo', 'w', 'e', 'r', 's', 't', '<unk>'`

4. **Repeat the process until desired granularity**:

Final vocabulary : `'low', 'lower', 'est', '<unk>'`

5. **The dictionary of vocabulary comprise of**:

`{'low':0, 'lower':1, 'est':3, '<unk>':3}`


We will apply a BPE tokenizer to our training and validation data. Specifically, we will use the tokenizer used for GPT2 models. Later we will compare the statisctics from the clean data w.r.t. the raw dataset.

</div>


In [ ]:

def preprocess_tokenizer_update(batch):
    return {'tokenized': tokenizer(batch['text'])['input_ids']}


_LOGGER.info('Tokenizing training dataset')
stories_train_dataset = stories_train_dataset.map(
    preprocess_tokenizer_update,
    batched=True,
    batch_size=tokenization_batch_size,
    num_proc=num_proc,
    desc='Tokenizing train data'
)

_LOGGER.info('Tokenizing validation dataset')
stories_valid_dataset = stories_valid_dataset.map(
    preprocess_tokenizer_update,
    batched=True,
    batch_size=tokenization_batch_size,
    num_proc=num_proc,
    desc='Tokenizing val data'
)

print('Done!')

## Save prepared data

Continue with scoring and filtering in [perplexity_filtering.ipynb](perplexity_filtering.ipynb).


In [ ]:
# Save after normalization, deduplication, and GPT-2 tokenization.
data_path.mkdir(parents=True, exist_ok=True)
stories_train_dataset.save_to_disk(data_path / 'tinystories_prepared_train')
stories_valid_dataset.save_to_disk(data_path / 'tinystories_prepared_validation')